<a href="https://colab.research.google.com/github/Iameeshan26/mlprojects/blob/main/buildmodel_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

[Learn the Basics](intro.html) \|\|
[Quickstart](quickstart_tutorial.html) \|\|
[Tensors](tensorqs_tutorial.html) \|\| [Datasets &
DataLoaders](data_tutorial.html) \|\|
[Transforms](transforms_tutorial.html) \|\| **Build Model** \|\|
[Autograd](autogradqs_tutorial.html) \|\|
[Optimization](optimization_tutorial.html) \|\| [Save & Load
Model](saveloadrun_tutorial.html)

Build the Neural Network
========================

Neural networks comprise of layers/modules that perform operations on
data. The [torch.nn](https://pytorch.org/docs/stable/nn.html) namespace
provides all the building blocks you need to build your own neural
network. Every module in PyTorch subclasses the
[nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html).
A neural network is a module itself that consists of other modules
(layers). This nested structure allows for building and managing complex
architectures easily.

In the following sections, we\'ll build a neural network to classify
images in the FashionMNIST dataset.


In [ ]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

Get Device for Training
=======================

We want to be able to train our model on an
[accelerator](https://pytorch.org/docs/stable/torch.html#accelerators)
such as CUDA, MPS, MTIA, or XPU. If the current accelerator is
available, we will use it. Otherwise, we use the CPU.


In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cpu device


Define the Class
================

We define our neural network by subclassing `nn.Module`, and initialize
the neural network layers in `__init__`. Every `nn.Module` subclass
implements the operations on input data in the `forward` method.


In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

We create an instance of `NeuralNetwork`, and move it to the `device`,
and print its structure.


In [ ]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


To use the model, we pass it the input data. This executes the model\'s
`forward`, along with some [background
operations](https://github.com/pytorch/pytorch/blob/270111b7b611d174967ed204776985cefca9c144/torch/nn/modules/module.py#L866).
Do not call `model.forward()` directly!

Calling the model on the input returns a 2-dimensional tensor with dim=0
corresponding to each output of 10 raw predicted values for each class,
and dim=1 corresponding to the individual values of each output. We get
the prediction probabilities by passing it through an instance of the
`nn.Softmax` module.


In [ ]:
X = torch.rand(1, 28, 28, device=device)
logits = model(X)
pred_probab = nn.Softmax(dim=1)(logits)
y_pred = pred_probab.argmax(1)
print(f"Predicted class: {y_pred}")

Predicted class: tensor([2])


------------------------------------------------------------------------


Model Layers
============

Let\'s break down the layers in the FashionMNIST model. To illustrate
it, we will take a sample minibatch of 3 images of size 28x28 and see
what happens to it as we pass it through the network.


In [ ]:
input_image = torch.rand(3,28,28)
print(input_image.size())
input_image

torch.Size([3, 28, 28])


tensor([[[0.5170, 0.1496, 0.4830,  ..., 0.4276, 0.7609, 0.6831],
         [0.8885, 0.5641, 0.7978,  ..., 0.7789, 0.6126, 0.2185],
         [0.6394, 0.0942, 0.5871,  ..., 0.7557, 0.5684, 0.1630],
         ...,
         [0.4607, 0.7395, 0.3091,  ..., 0.8131, 0.1422, 0.0683],
         [0.8310, 0.5835, 0.0773,  ..., 0.3437, 0.3915, 0.1384],
         [0.2392, 0.4847, 0.7650,  ..., 0.4549, 0.7385, 0.6384]],

        [[0.4066, 0.8014, 0.1747,  ..., 0.7044, 0.7047, 0.0953],
         [0.5970, 0.9646, 0.8509,  ..., 0.9683, 0.2517, 0.2256],
         [0.3478, 0.8122, 0.8528,  ..., 0.2219, 0.2266, 0.2028],
         ...,
         [0.6116, 0.3446, 0.0405,  ..., 0.4248, 0.8290, 0.7715],
         [0.8357, 0.5132, 0.7002,  ..., 0.4735, 0.5910, 0.5280],
         [0.9307, 0.1048, 0.8130,  ..., 0.1980, 0.0552, 0.6860]],

        [[0.5279, 0.4068, 0.5105,  ..., 0.3282, 0.9515, 0.6335],
         [0.4936, 0.1704, 0.0397,  ..., 0.6779, 0.3344, 0.9437],
         [0.3114, 0.4738, 0.5215,  ..., 0.7685, 0.7375, 0.

nn.Flatten
==========

We initialize the
[nn.Flatten](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html)
layer to convert each 2D 28x28 image into a contiguous array of 784
pixel values ( the minibatch dimension (at dim=0) is maintained).


In [ ]:
flatten = nn.Flatten()
flat_image = flatten(input_image)
print(flat_image.size())
flat_image

torch.Size([3, 784])


tensor([[0.5170, 0.1496, 0.4830,  ..., 0.4549, 0.7385, 0.6384],
        [0.4066, 0.8014, 0.1747,  ..., 0.1980, 0.0552, 0.6860],
        [0.5279, 0.4068, 0.5105,  ..., 0.3389, 0.9901, 0.3087]])

nn.Linear
=========

The [linear
layer](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html)
is a module that applies a linear transformation on the input using its
stored weights and biases.


In [ ]:
layer1 = nn.Linear(in_features=28*28, out_features=20)
hidden1 = layer1(flat_image)
print(hidden1.size())
layer1

torch.Size([3, 20])


Linear(in_features=784, out_features=20, bias=True)

nn.ReLU
=======

Non-linear activations are what create the complex mappings between the
model\'s inputs and outputs. They are applied after linear
transformations to introduce *nonlinearity*, helping neural networks
learn a wide variety of phenomena.

In this model, we use
[nn.ReLU](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)
between our linear layers, but there\'s other activations to introduce
non-linearity in your model.


In [ ]:
print(f"Before ReLU: {hidden1}\n\n")
hidden1 = nn.ReLU()(hidden1)
print(f"After ReLU: {hidden1}")

Before ReLU: tensor([[ 0.3563,  0.0371,  0.6212, -0.2472,  0.0578, -0.4271,  0.3734,  0.6899,
          0.1189, -0.1234,  0.0524, -0.2080,  0.5157, -0.2309,  0.1022, -0.3238,
         -0.6169, -0.3364,  0.2468, -0.3904],
        [ 0.2218,  0.4552,  0.7933, -0.2593,  0.0688, -0.3744,  0.1495,  0.8191,
          0.3161, -0.2737,  0.3645, -0.3815,  0.5133, -0.2683,  0.1363, -0.4546,
         -0.4446, -0.4730,  0.0480, -0.2795],
        [ 0.3466, -0.0447,  0.4642, -0.0861,  0.1642, -0.1396,  0.1037,  0.8837,
          0.2366,  0.3243,  0.1334, -0.0856,  0.4740, -0.2546,  0.1380, -0.3115,
         -0.6088,  0.1285,  0.0200, -0.2663]], grad_fn=<AddmmBackward0>)


After ReLU: tensor([[0.3563, 0.0371, 0.6212, 0.0000, 0.0578, 0.0000, 0.3734, 0.6899, 0.1189,
         0.0000, 0.0524, 0.0000, 0.5157, 0.0000, 0.1022, 0.0000, 0.0000, 0.0000,
         0.2468, 0.0000],
        [0.2218, 0.4552, 0.7933, 0.0000, 0.0688, 0.0000, 0.1495, 0.8191, 0.3161,
         0.0000, 0.3645, 0.0000, 0.5133, 0.0000, 0.13

nn.Sequential
=============

[nn.Sequential](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html)
is an ordered container of modules. The data is passed through all the
modules in the same order as defined. You can use sequential containers
to put together a quick network like `seq_modules`.


In [ ]:
seq_modules = nn.Sequential(
    flatten,
    layer1,
    nn.ReLU(),
    nn.Linear(20, 10)
)
input_image = torch.rand(3,28,28)
logits = seq_modules(input_image)

nn.Softmax
==========

The last linear layer of the neural network returns [logits]{.title-ref}
- raw values in \[-infty, infty\] - which are passed to the
[nn.Softmax](https://pytorch.org/docs/stable/generated/torch.nn.Softmax.html)
module. The logits are scaled to values \[0, 1\] representing the
model\'s predicted probabilities for each class. `dim` parameter
indicates the dimension along which the values must sum to 1.


In [ ]:
softmax = nn.Softmax(dim=1)
pred_probab = softmax(logits)

Model Parameters
================

Many layers inside a neural network are *parameterized*, i.e. have
associated weights and biases that are optimized during training.
Subclassing `nn.Module` automatically tracks all fields defined inside
your model object, and makes all parameters accessible using your
model\'s `parameters()` or `named_parameters()` methods.

In this example, we iterate over each parameter, and print its size and
a preview of its values.


In [ ]:
print(f"Model structure: {model}\n\n")

for name, param in model.named_parameters():
    print(f"Layer: {name} | Size: {param.size()} | Values : {param[:2]} \n")

Model structure: NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Layer: linear_relu_stack.0.weight | Size: torch.Size([512, 784]) | Values : tensor([[ 0.0031,  0.0039,  0.0042,  ..., -0.0328,  0.0126,  0.0079],
        [-0.0339,  0.0249, -0.0278,  ..., -0.0030,  0.0120, -0.0245]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.0.bias | Size: torch.Size([512]) | Values : tensor([0.0266, 0.0036], grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.weight | Size: torch.Size([512, 512]) | Values : tensor([[ 0.0202, -0.0415,  0.0382,  ..., -0.0221, -0.0128, -0.0076],
        [ 0.0306, -0.0019,  0.0325,  ...,  0.0022, -0.0019,  0.0112]],
       grad_fn=<SliceBackward0>) 

Layer: linear_relu_stack.2.bias | Si

------------------------------------------------------------------------


Further Reading
===============

-   [torch.nn API](https://pytorch.org/docs/stable/nn.html)
